# The V2 OrbitNet Agent: A Complete Walkthrough

This notebook dissects every component of the **V2 OrbitNet** agent for
[Kaggle Orbit Wars](https://www.kaggle.com/competitions/orbit-wars). We use a
**real Kaggle environment** (not mock data) so you see exactly what the agent sees.

**What you'll learn:**
1. How raw Kaggle observations become a 40×22 planet feature matrix
2. Fleet destination prediction — figuring out where every fleet is heading
3. The OrbitNet architecture (~518K params): self-attention + pairwise output head
4. How a 40×41 allocation matrix turns into game moves
5. Dense relative reward and early production bonus
6. PPO training with mixed self-play scheduling

### V1 vs V2: The Key Difference

| | V1 (Transformer Mixed PPO) | V2 (OrbitNet) |
|---|---|---|
| Forward passes per step | N (one per owned planet) | **1** (all planets at once) |
| Action space | Target + fraction per planet | **40×41 allocation matrix** |
| Ship fraction | Discrete 5-way (0.2–1.0) | **Softmax probability** |
| Cross-planet coordination | Sequential (transit updates) | **Self-attention** |

The core insight: process ALL planets simultaneously. One forward pass outputs
an allocation matrix where each owned planet distributes ships to all valid targets.
The softmax probability is both the sampling distribution (for PPO) and the ship
fraction (for execution).

## Part 0: Setup

In [ ]:
#@title Install Dependencies & Clone Repo (Colab only)
#@markdown Run this cell if you're on Google Colab. Skip if running locally.

import subprocess, importlib, os
try:
    importlib.import_module("kaggle_environments")
    print("Dependencies already installed.")
except ImportError:
    subprocess.check_call([
        "pip", "install", "-q",
        "kaggle-environments>=1.28.0",
        "torch", "pyyaml", "numpy",
    ])
    print("Installed successfully.")

# Clone repo if not already present
_ow_path = "/tmp/ow"
if not os.path.exists(os.path.join(_ow_path, "v2", "config.py")):
    if os.path.exists(_ow_path):
        subprocess.check_call(["rm", "-rf", _ow_path])
    subprocess.check_call(["git", "clone", "https://github.com/oshbocker/orbit_wars.git", _ow_path])
    print(f"Cloned repo to {_ow_path}")
else:
    print(f"Repo already at {_ow_path}")
os.chdir(_ow_path)

In [ ]:
import sys, os
from pathlib import Path

# Ensure repo root is on path (handles both local and Colab)
_repo_root = None
_nb_dir = Path(os.path.abspath('')).resolve()
for _p in [_nb_dir, _nb_dir.parent, *_nb_dir.parents]:
    if (_p / 'v2' / 'config.py').exists():
        _repo_root = str(_p)
        break

if _repo_root is None:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    _repo_root = os.getcwd()

if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)
os.chdir(_repo_root)

import math
import numpy as np
import torch
import torch.nn.functional as F

from v2.config import load_v2_config
from v2.model import OrbitNet
from v2.features import encode_features, PLANET_FEAT_DIM, GLOBAL_FEAT_DIM
from v2.state import predict_fleet_destination, compute_incoming_fleets
from v2.actions import sample_actions, decode_actions, action_log_prob_and_entropy
from v2.reward import compute_reward
from src.game_types import parse_observation, PlanetState, FleetState, GameState
from src.features import fleet_speed, passes_through_sun

cfg = load_v2_config('configs/v2_default.yaml')
device = torch.device('cpu')

print(f"Config loaded: {cfg.run_name}")
print(f"  Reward mode: {cfg.reward.reward_mode}")
print(f"  Max planets: {cfg.env.max_planets}")
print(f"  Model: embed_dim={cfg.model.embed_dim}, layers={cfg.model.n_layers}, heads={cfg.model.n_heads}")
print(f"  Allocation threshold: {cfg.env.allocation_threshold}")

---

## Part 1: The Game State — What the Agent Sees

Before any neural network, we need to understand the raw observation.
Let's create a real Kaggle environment and inspect what arrives.

In [ ]:
# Create a real Kaggle orbit_wars environment
from kaggle_environments import make

env = make("orbit_wars", configuration={"seed": 42}, debug=False)
env.reset(num_agents=2)
# Take one empty step to get the initial observation
states = env.step([[], []])
obs = states[0].observation

# Parse into our typed GameState
state = parse_observation(obs)

print(f"Step: {state.step}")
print(f"Player: {state.player}")
print(f"Angular velocity: {state.angular_velocity:.4f} rad/turn")
print(f"Planets: {len(state.planets)}")
print(f"Fleets: {len(state.fleets)}")

In [ ]:
# Inspect every planet
print(f"{'ID':>3} {'Owner':>5} {'X':>6} {'Y':>6} {'R':>5} {'Ships':>6} {'Prod':>5} {'Orbit':>5}")
print("-" * 50)
for p in state.planets:
    orbit_str = f"{p.orbital_radius:.1f}" if p.is_orbiting else "-"
    own_str = {state.player: 'MINE', -1: 'neut'}.get(p.owner, f'E{p.owner}')
    print(f"{p.id:3d} {own_str:>5} {p.x:6.1f} {p.y:6.1f} {p.radius:5.2f} {p.ships:6d} {p.production:5d} {orbit_str:>5}")

orbiting = [p for p in state.planets if p.is_orbiting]
static = [p for p in state.planets if not p.is_orbiting]
mine = [p for p in state.planets if p.owner == state.player]
print(f"\n{len(orbiting)} orbiting, {len(static)} static, {len(mine)} owned by us")

### 1.1 Fleet Physics Refresher

Fleet speed is a nonlinear function of ship count — small fleets crawl, large fleets
race. This asymmetry is critical for strategy.

In [ ]:
ship_counts = [1, 5, 10, 25, 50, 100, 200, 500, 1000]
print("Ships -> Speed (units/turn) -> Time to cross board (100 units)")
print("-" * 55)
for s in ship_counts:
    sp = fleet_speed(s)
    turns = 100 / sp
    bar = '#' * int(sp * 5)
    print(f"  {s:5d} ships -> {sp:.2f} u/t -> {turns:.0f} turns  {bar}")

print(f"\nKey insight: 1 ship takes 100 turns to cross the board.")
print(f"             500 ships take only {100/fleet_speed(500):.0f} turns.")
print(f"             Don't send tiny fleets to distant targets!")

---

## Part 2: Fleet Destination Prediction — Where Is Every Fleet Going?

Before encoding features, V2 predicts which planet each in-flight fleet will hit.
This is the **foundation** of the feature encoding: for each planet, we know exactly
how many ships from each team are heading toward it and when they'll arrive.

Two challenges:
- **Static planets**: Direct ray-circle intersection (fast, exact)
- **Orbiting planets**: Step-by-step forward simulation (the planet moves!)

In [ ]:
# Play a few turns to generate some fleets
from agents.apex import agent as apex_agent

for _ in range(5):
    moves_p0 = apex_agent(states[0].observation)
    moves_p1 = apex_agent(states[1].observation)
    states = env.step([moves_p0, moves_p1])

obs = states[0].observation
state = parse_observation(obs)

print(f"After {state.step} steps: {len(state.fleets)} fleets in flight")
print()

if state.fleets:
    print(f"{'ID':>3} {'Owner':>5} {'X':>6} {'Y':>6} {'Angle':>6} {'Ships':>6} {'From':>4}")
    print("-" * 42)
    for f in state.fleets[:10]:
        own_str = 'MINE' if f.owner == state.player else f'E{f.owner}'
        print(f"{f.id:3d} {own_str:>5} {f.x:6.1f} {f.y:6.1f} {f.angle:6.2f} {f.ships:6d} {f.from_planet_id:4d}")

In [ ]:
# Predict destinations for every fleet
print("=== Fleet Destination Predictions ===")
print()

for f in state.fleets[:8]:
    target, eta = predict_fleet_destination(
        f, state.planets, state.step, state.angular_velocity
    )
    own_str = 'ours' if f.owner == state.player else f'enemy(P{f.owner})'
    if target is not None:
        tgt_own = {state.player: 'our', -1: 'neutral'}.get(target.owner, f'enemy')
        print(f"  Fleet {f.id} ({f.ships} ships, {own_str}): "
              f"-> Planet {target.id} ({tgt_own}, {target.ships} ships) "
              f"ETA={eta:.1f} turns")
    else:
        print(f"  Fleet {f.id} ({f.ships} ships, {own_str}): "
              f"-> NOWHERE (sun/edge/miss)")

In [ ]:
# Aggregate: incoming fleet info per planet
incoming = compute_incoming_fleets(state, state.player)

print("=== Incoming Fleets Per Planet ===")
print(f"{'Planet':>6} {'Own Ships':>10} {'Own ETA':>8} {'E1 Ships':>10} {'E1 ETA':>8}")
print("-" * 50)

for pid, info in sorted(incoming.items()):
    planet = state.planets_by_id.get(pid)
    if planet is None:
        continue
    own_str = {state.player: '*', -1: ' '}.get(planet.owner, 'E')
    print(f"  P{pid:3d}{own_str} {info.ships[0]:10.0f} {info.eta[0]:8.1f} "
          f"{info.ships[1]:10.0f} {info.eta[1]:8.1f}")

print(f"\n{len(incoming)} planets have incoming fleets")
print("(* = we own this planet)")

> **Why this matters:** In V1, fleet transit was computed per-planet and updated
> sequentially as each source planet made decisions. V2 computes all fleet
> destinations **once** and bakes them into the feature matrix. The self-attention
> mechanism replaces the sequential transit updates — all planets see all incoming
> fleets simultaneously.

---

## Part 3: Feature Encoding — The 40×22 Planet Matrix

V2's feature encoding is radically different from V1. Instead of building separate
features per source-target pair, we encode **every planet** into a fixed-size matrix.

```
V1: For each owned planet:
      global[9] + source[2+7] + knn[3×(2+4)] + targets[30×(2+11)] = ~400 features
      × N owned planets = N forward passes

V2: One matrix for ALL planets:
      planets[40×22] + global[8] = 888 features
      × 1 forward pass = done
```

The magic: the model learns which planets are sources and which are targets through
the **ownership features** (columns 2–5) and the **own_mask**.

In [ ]:
# Encode the current game state
features = encode_features(state, cfg.env)

print("=== V2Features Shapes ===")
print(f"  planet_features: {features.planet_features.shape}  (max_planets × feature_dim)")
print(f"  global_features: {features.global_features.shape}  (game-level context)")
print(f"  planet_mask:     {features.planet_mask.shape}  ({features.planet_mask.sum()} planets exist)")
print(f"  own_mask:        {features.own_mask.shape}  ({features.own_mask.sum()} planets owned)")
print(f"  planet_ids:      {sum(1 for x in features.planet_ids if x >= 0)} valid IDs")

In [ ]:
# Show the 22 features for each planet
FEAT_NAMES = [
    'exists', 'orbiting',
    'own_mine', 'own_e1', 'own_e2', 'own_e3',
    'ships', 'x', 'y', 'dist_center', 'sin(θ)', 'cos(θ)',
    'production', 'radius',
    'in_own_ships', 'in_e1_ships', 'in_e2_ships', 'in_e3_ships',
    'in_own_eta', 'in_e1_eta', 'in_e2_eta', 'in_e3_eta',
]

print("=== Per-Planet Feature Vector (22 dimensions) ===")
print()

# Show features for a planet we own and an enemy planet
my_slot = next(i for i in range(40) if features.own_mask[i])
enemy_slots = [i for i in range(40) if features.planet_mask[i] and not features.own_mask[i]
               and features.planet_features[i, 4] == 1.0]  # enemy owned
enemy_slot = enemy_slots[0] if enemy_slots else None

for slot, label in [(my_slot, 'OUR PLANET'), (enemy_slot, 'ENEMY PLANET')]:
    if slot is None:
        continue
    pf = features.planet_features[slot]
    p = features.planet_states[slot]
    print(f"--- {label} (slot {slot}, Planet ID {features.planet_ids[slot]}) ---")
    for j, (name, val) in enumerate(zip(FEAT_NAMES, pf)):
        raw = ""
        if name == 'ships' and p:
            raw = f"  (raw: {p.ships} ships)"
        elif name == 'x' and p:
            raw = f"  (raw: {p.x:.1f})"
        elif name == 'y' and p:
            raw = f"  (raw: {p.y:.1f})"
        elif name == 'production' and p:
            raw = f"  (raw: {p.production})"
        print(f"  [{j:2d}] {name:15s} = {val:+.4f}{raw}")
    print()

### 3.1 Feature Design Choices

Several deliberate decisions:

- **Planet ID = matrix row**: Planet with `id=5` always goes in row 5. This gives
  the model a **stable reference frame** within a game. No sorting or reordering.

- **Ownership as one-hot** (columns 2–5): Rather than a single owner ID, we encode
  `[mine, enemy1, enemy2, enemy3]`. This makes ownership **immediately visible**
  to the network without needing to learn numerical player ID semantics.

- **Polar coordinates** (columns 9–11): `dist_from_center`, `sin(θ)`, `cos(θ)` capture
  the radial structure of the board. Orbiting planets move in circles — polar
  features make this natural.

- **Incoming fleets per team** (columns 14–21): Four teams' incoming ships and ETAs.
  This is the output of `compute_incoming_fleets()` from Part 2. The model sees
  reinforcement and threat information for every planet simultaneously.

- **Log normalization** for ships (`log(1+ships)/7`): Ship counts span 0–1000+.
  Log compression keeps features in a neural-network-friendly range.

In [ ]:
# Global features: the game-level context vector
GLOBAL_NAMES = [
    'step / 500',
    'angular_vel / 0.05',
    'own ships (log)',
    'best enemy ships (log)',
    'own prod fraction',
    'enemy prod fraction',
    'own planets / 40',
    'total planets / 40',
]

print("=== Global Features (8 dimensions) ===")
print("These are broadcast-added to every planet's embedding.")
print()
for i, (name, val) in enumerate(zip(GLOBAL_NAMES, features.global_features)):
    print(f"  [{i}] {name:25s} = {val:.4f}")

print(f"\nKey ratio: own_prod={features.global_features[4]:.3f} vs "
      f"enemy_prod={features.global_features[5]:.3f}")
print(f"We {'lead' if features.global_features[4] > features.global_features[5] else 'trail'} "
      f"in production.")

In [ ]:
# Visualize the planet matrix structure
print("=== Planet Matrix Overview ===")
print(f"Shape: {features.planet_features.shape}")
print(f"Non-zero rows (existing planets): {features.planet_mask.sum()}")
print(f"Zero rows (empty slots): {(~features.planet_mask).sum()}")
print()

# Show which slots are occupied
print("Slot map (. = empty, O = ours, E = enemy, N = neutral):")
row = ""
for i in range(40):
    if not features.planet_mask[i]:
        row += "."
    elif features.own_mask[i]:
        row += "O"
    elif features.planet_features[i, 4] == 1.0:  # enemy1
        row += "E"
    else:
        row += "N"
    if (i + 1) % 10 == 0:
        row += " "
print(f"  [{row.strip()}]")
print(f"  Slots 0-9       10-19      20-29      30-39")
print(f"\nFeature value range: [{features.planet_features.min():.3f}, {features.planet_features.max():.3f}]")
print(f"Global value range:  [{features.global_features.min():.3f}, {features.global_features.max():.3f}]")

---

## Part 4: The OrbitNet Architecture — One Pass to Rule Them All

OrbitNet processes all 40 planet slots simultaneously and outputs a 40×41 allocation
matrix. The architecture has 5 stages:

```
Planet features [40×22] + Global features [8]
    → Planet Embedding (MLP + LayerNorm)          [40×128]
    → + Global Embedding (broadcast add)           [40×128]
    → Self-Attention Encoder (3 layers, 4 heads)  [40×128]
    → Pairwise Output Head                        [40×41]
    → + Value Head (masked mean pool → MLP)       scalar
```

> **Architecture philosophy**: Unlike V1 which uses a separate forward pass per
> planet, OrbitNet lets planets **talk to each other** through self-attention.
> Planet A can see that Planet B is already sending ships to target C, and adjust
> its own allocation accordingly. This is cross-planet coordination for free.

In [ ]:
# Create the model
model = OrbitNet(cfg.model).to(device)
model.eval()

# Count parameters by component
def count_params(module):
    return sum(p.numel() for p in module.parameters())

print("=== OrbitNet Architecture ===")
print(f"Total parameters: {count_params(model):,}\n")

components = [
    ('planet_embed (MLP + LN)', model.planet_embed),
    ('global_embed (MLP)', model.global_embed),
    ('transformer_blocks (×3)', model.transformer_blocks),
    ('final_ln', model.final_ln),
    ('src_proj', model.src_proj),
    ('tgt_proj', model.tgt_proj),
    ('pair_mlp', model.pair_mlp),
    ('hold_head', model.hold_head),
    ('value_head', model.value_head),
]

for name, mod in components:
    n = count_params(mod)
    pct = 100 * n / count_params(model)
    bar = '#' * int(pct / 2)
    print(f"  {name:30s} {n:>8,} params ({pct:5.1f}%)  {bar}")

### 4.1 Stage 1: Planet Embedding

Each planet's 22-dimensional feature vector is projected into the shared 128-dimensional
embedding space via a 2-layer MLP with LayerNorm.

The global 8-dimensional vector gets its own MLP and is **broadcast-added** to every
planet embedding. This means every planet knows the game-level context (step, ship
advantage, production balance) without wasting attention capacity on it.

In [ ]:
# Prepare tensors (batch size 1)
B = 1
pf_t = torch.from_numpy(features.planet_features).unsqueeze(0)    # [1, 40, 22]
gf_t = torch.from_numpy(features.global_features).unsqueeze(0)    # [1, 8]
pm_t = torch.from_numpy(features.planet_mask).unsqueeze(0)        # [1, 40]
om_t = torch.from_numpy(features.own_mask).unsqueeze(0)           # [1, 40]

d = cfg.model.embed_dim
P = cfg.env.max_planets

with torch.no_grad():
    # Planet embedding
    planet_emb = model.planet_embed(pf_t)   # [1, 40, 128]
    print(f"Planet embedding:  {pf_t.shape} -> {planet_emb.shape}")
    print(f"  Embedding norms (existing planets): "
          f"min={planet_emb[0, pm_t[0]].norm(dim=-1).min():.2f}, "
          f"max={planet_emb[0, pm_t[0]].norm(dim=-1).max():.2f}")
    print(f"  Embedding norms (empty slots):      "
          f"min={planet_emb[0, ~pm_t[0]].norm(dim=-1).min():.2f}, "
          f"max={planet_emb[0, ~pm_t[0]].norm(dim=-1).max():.2f}")
    print()

    # Global embedding (broadcast add)
    global_emb = model.global_embed(gf_t)   # [1, 128]
    x = planet_emb + global_emb.unsqueeze(1)  # [1, 40, 128]
    print(f"Global embedding:  {gf_t.shape} -> {global_emb.shape}")
    print(f"After broadcast add: {x.shape}")
    print(f"\n> Every planet now knows: step={features.global_features[0]:.3f}, "
          f"own_ships={features.global_features[2]:.3f}, etc.")

### 4.2 Stage 2: Self-Attention Encoder

3 layers of pre-LayerNorm self-attention (4 heads each) process the planet sequence.
The `key_padding_mask` ensures empty slots don't participate in attention.

This is where the magic happens: planets **attend to each other**. An owned planet
can see enemy fleet concentrations on nearby planets, production values across the
map, and what other owned planets are positioned to do.

In [ ]:
with torch.no_grad():
    key_padding_mask = ~pm_t  # True = ignore this position

    print(f"Key padding mask: {key_padding_mask.shape}")
    print(f"  Masked (empty) slots: {key_padding_mask.sum().item()}")
    print(f"  Active slots: {(~key_padding_mask).sum().item()}")
    print()

    # Step through each transformer block
    for i, block in enumerate(model.transformer_blocks):
        x_before = x.clone()
        x = block(x, key_padding_mask=key_padding_mask)
        delta = (x - x_before).norm(dim=-1).mean().item()
        print(f"  Block {i}: {x.shape}  ||delta|| = {delta:.4f}")

    x = model.final_ln(x)
    print(f"\nFinal encoded: {x.shape}")
    print(f"  Encoded norms: min={x[0, pm_t[0]].norm(dim=-1).min():.2f}, "
          f"max={x[0, pm_t[0]].norm(dim=-1).max():.2f}")

In [ ]:
# Peek at attention patterns from the first block
with torch.no_grad():
    first_block = model.transformer_blocks[0]
    normed = first_block.ln1(planet_emb + global_emb.unsqueeze(1))
    _, attn_weights = first_block.attn(
        normed, normed, normed,
        key_padding_mask=key_padding_mask,
        need_weights=True, average_attn_weights=False
    )
    # attn_weights: [1, n_heads, 40, 40]

n_heads = attn_weights.shape[1]
active = pm_t[0].numpy()
active_ids = [i for i in range(40) if active[i]]

print(f"Attention weights: {attn_weights.shape} (batch, heads, queries, keys)")
print(f"Active planets: {len(active_ids)}")
print()

# Show attention from our owned planet to other planets
our_slot = next(i for i in range(40) if features.own_mask[i])
print(f"Attention FROM our planet (slot {our_slot}) TO other planets:")
print(f"{'Head':>5}  {'Top-3 attended planets':>40}")
print("-" * 50)
for h in range(n_heads):
    weights = attn_weights[0, h, our_slot].numpy()
    top3 = sorted(active_ids, key=lambda i: -weights[i])[:3]
    parts = []
    for idx in top3:
        p = features.planet_states[idx]
        own_str = {state.player: 'own', -1: 'N'}.get(p.owner, 'E') if p else '?'
        parts.append(f"P{features.planet_ids[idx]}({own_str})={weights[idx]:.3f}")
    print(f"  H{h}:  {', '.join(parts)}")

### 4.3 Stage 3: Pairwise Output Head

This is the core innovation of OrbitNet. Instead of outputting per-planet logits
independently, we compute **pairwise** logits between every source-target pair.

```
encoded [40×128]
    → src_proj [40×128]  ×  tgt_proj [40×128]
    → concat pairwise [40×40×256]
    → pair_mlp [40×40×1] → squeeze → pair_logits [40×40]
    → + hold_logits [40×1]
    → cat → logits [40×41]
```

Column 0 = "hold" (send nothing). Columns 1–40 = send to planet 0–39.

**Why pairwise?** A simple per-planet head can't express "Planet A should send to
Planet B but not C." The pairwise MLP sees both source and target representations
and can learn distance-aware, ownership-aware, threat-aware targeting.

In [ ]:
with torch.no_grad():
    # Full forward pass
    output = model(pf_t, gf_t, pm_t, om_t)

logits = output.logits  # [1, 40, 41]
value = output.value    # [1]

print(f"=== OrbitNet Output ===")
print(f"  logits: {logits.shape}  (planets × [hold + targets])")
print(f"  value:  {value.item():.4f}")
print()

# Masking verification
NEG_INF = torch.finfo(logits.dtype).min

# Non-owned source rows should be all -inf
non_owned = ~om_t[0]
non_owned_all_inf = (logits[0, non_owned] == NEG_INF).all().item()
print(f"Non-owned sources all -inf: {non_owned_all_inf}")

# Non-existent target columns should be -inf
non_exist = ~pm_t[0]
non_exist_targets_inf = (logits[0, :, 1:][:, non_exist] == NEG_INF).all().item()
print(f"Non-existent targets all -inf: {non_exist_targets_inf}")

# Self-targeting diagonal should be -inf
diag_inf = all(logits[0, i, i+1] == NEG_INF for i in range(40) if om_t[0, i])
print(f"Self-targeting diagonal all -inf: {diag_inf}")

In [ ]:
# Show the allocation for our owned planet
our_slot = next(i for i in range(40) if features.own_mask[i])
row_logits = logits[0, our_slot]  # [41]

# Filter to valid targets
valid_mask = torch.isfinite(row_logits)
probs = torch.softmax(row_logits, dim=-1)

p = features.planet_states[our_slot]
print(f"=== Allocation from Planet {features.planet_ids[our_slot]} "
      f"({p.ships} ships, prod={p.production}) ===")
print(f"{'Action':>20} {'Logit':>8} {'Prob':>8} {'Ships':>6}")
print("-" * 48)

# Sort by probability
indices = torch.argsort(probs, descending=True)
for idx in indices[:10]:
    idx = idx.item()
    if not valid_mask[idx]:
        continue
    prob = probs[idx].item()
    logit = row_logits[idx].item()
    ships = int(math.floor(p.ships * prob))

    if idx == 0:
        label = "HOLD"
    else:
        tgt = features.planet_states[idx - 1]
        if tgt:
            own_str = {state.player: 'own', -1: 'N'}.get(tgt.owner, 'E')
            label = f"-> P{tgt.id} ({own_str}, {tgt.ships}sh)"
        else:
            label = f"-> slot {idx-1}"

    bar = '#' * int(prob * 40)
    print(f"{label:>20} {logit:8.3f} {prob:8.3f} {ships:6d}  {bar}")

print(f"\n> The softmax probability IS the ship fraction.")
print(f"  P(hold)={probs[0].item():.3f} means {int(p.ships * probs[0].item())} ships stay put.")

### 4.4 The Value Head

The value head predicts how good the current state is for us. It uses **masked mean
pooling** over existing planets (not all 40 slots) to get a single representation,
then passes it through a 2-layer MLP.

```
encoded [40×128] × planet_mask [40] → mean pool [128] → MLP → value (scalar)
```

The value is initialized near 0 (zero-init on the final linear layer) and learns
to predict discounted future reward.

In [ ]:
print(f"Value prediction: {value.item():.4f}")
print(f"  (Untrained model: should be near 0 due to zero-init)")
print()
print(f"Value head architecture:")
print(f"  masked_mean_pool({features.planet_mask.sum()} planets) [128]")
print(f"  -> Linear(128, 128) + ReLU")
print(f"  -> Linear(128, 1)")
print(f"  -> scalar")

### 4.5 Zero-Init for Stable Training

Three output heads are **zero-initialized** (final layer weights and biases set to 0):

| Head | Zero-init effect | Why |
|------|-----------------|-----|
| `pair_mlp` | All pair logits = 0 | Uniform softmax → random targeting → safe exploration |
| `hold_head` | Hold logit = 0 | Equal chance of holding vs sending → tries both |
| `value_head` | Value = 0 | No initial bias → learns from reward signal cleanly |

Without zero-init, random initial weights create strong arbitrary preferences
that PPO wastes early updates overcoming.

---

## Part 5: Actions — From Allocation Matrix to Game Moves

The 40×41 logit matrix becomes actual Kaggle move commands through two modes:

**Training (stochastic):** For each owned planet, sample ONE target from
`Categorical(logits)`. Send `floor(ships × P(target))` ships. The log probability
is the sum over all owned planets.

**Evaluation (deterministic):** Use the full softmax. For each target with
`P > threshold` (default 0.05), send `floor(ships × P)` ships. This allows
sending to **multiple targets simultaneously** — a natural consequence of softmax.

> **Elegant unification:** The softmax probability serves triple duty:
> 1. Sampling distribution for PPO exploration
> 2. Ship fraction (how many ships to send)
> 3. Multi-target allocation (eval mode)

In [ ]:
# Stochastic sampling (training mode)
with torch.no_grad():
    sampled = sample_actions(output, om_t, deterministic=False)

print("=== Stochastic Sampling (Training) ===")
print(f"  target_indices: {sampled.target_indices.shape}  (one per planet slot)")
print(f"  log_prob:       {sampled.log_prob.item():.4f}  (sum over owned planets)")
print(f"  entropy:        {sampled.entropy.item():.4f}  (sum over owned planets)")
print()

# Show sampled action for our planet
our_action = sampled.target_indices[0, our_slot].item()
if our_action == 0:
    print(f"  Planet {features.planet_ids[our_slot]} sampled: HOLD")
else:
    tgt = features.planet_states[our_action - 1]
    prob = probs[our_action].item()
    ships = int(math.floor(p.ships * prob))
    if tgt:
        print(f"  Planet {features.planet_ids[our_slot]} sampled: "
              f"send {ships} ships to Planet {tgt.id} (P={prob:.3f})")
    else:
        print(f"  Planet {features.planet_ids[our_slot]} sampled: "
              f"target slot {our_action - 1} (P={prob:.3f})")

In [ ]:
# Deterministic decoding (eval mode)
moves = decode_actions(output, features, state, cfg.env, deterministic=True)

print("=== Deterministic Decoding (Evaluation) ===")
print(f"  Allocation threshold: {cfg.env.allocation_threshold}")
print(f"  Moves generated: {len(moves)}")
print()

for move in moves:
    src_id, angle, ships = move[0], move[1], move[2]
    src_p = state.planets_by_id.get(int(src_id))
    print(f"  Planet {int(src_id)} ({src_p.ships if src_p else '?'} ships) "
          f"-> angle={angle:.2f} rad, {int(ships)} ships")

if not moves:
    print("  (No moves above threshold — untrained model sends nothing)")

# Compare: stochastic sampling
stoch_moves = decode_actions(output, features, state, cfg.env, deterministic=False)
print(f"\n  Stochastic mode: {len(stoch_moves)} moves (samples ONE target per planet)")

In [ ]:
# Demonstrate the log_prob recomputation (used in PPO update)
with torch.no_grad():
    new_lp, new_ent = action_log_prob_and_entropy(
        output, om_t, sampled.target_indices
    )

print("=== Log-Prob Recomputation (for PPO importance ratio) ===")
print(f"  Original log_prob:    {sampled.log_prob.item():.4f}")
print(f"  Recomputed log_prob:  {new_lp.item():.4f}")
print(f"  Match: {torch.allclose(sampled.log_prob, new_lp)}")
print()
print(f"  Original entropy:     {sampled.entropy.item():.4f}")
print(f"  Recomputed entropy:   {new_ent.item():.4f}")
print(f"  Match: {torch.allclose(sampled.entropy, new_ent)}")
print()
print("  > PPO uses recomputed log_prob to calculate the importance ratio:")
print("    ratio = exp(new_log_prob - old_log_prob)")
print("    If ratio ≈ 1: policy hasn't changed much (good)")
print("    If ratio >> 1 or << 1: clipping kicks in (prevents catastrophic updates)")

---

## Part 6: Dense Relative Reward — Learning to Win, Not Just Accumulate

The reward function drives everything. V2 uses the same `dense_relative` reward
as V1, but computed as a standalone function.

```
reward = Δ(own_ships − best_enemy_ships) × ship_coef
       + Δ(own_prod  − best_enemy_prod)  × prod_coef × prod_multiplier
       + terminal (±1 at game end)
```

The **relative** framing is crucial: gaining 10 ships doesn't help if the enemy
gained 20. Only improvements to the *gap* are rewarded.

In [ ]:
print("=== Reward Mode Comparison ===\n")

print("1. SPARSE (terminal only):")
print("   Mid-game: reward = 0 (always)")
print("   End-game: reward = +1 (win) or -1 (loss)")
print("   Problem: 500 steps of zeros → credit assignment nightmare")
print()
print("2. DENSE ABSOLUTE:")
print("   reward = Δ(own_ships) × 0.002 + Δ(own_prod) × 0.005")
print("   Problem: rewards hoarding — never attack if it costs ships")
print()
print("3. DENSE RELATIVE (V2 default):")
print(f"   reward = Δ(ship_gap) × {cfg.reward.dense_ship_coef} "
      f"+ Δ(prod_gap) × {cfg.reward.dense_prod_coef} × prod_mult")
print("   Rewards gaining ADVANTAGE, not just accumulating")
print("   Attacking is good if we gain more than we lose!")

In [ ]:
# Visualize the early production bonus schedule
bonus = cfg.reward.early_prod_bonus  # 9.0
bonus_steps = cfg.reward.early_prod_bonus_steps  # 50

print(f"=== Early Production Bonus ===")
print(f"  Bonus: {bonus} (multiplier at step 0 = {1 + bonus}×)")
print(f"  Decay: linear over {bonus_steps} steps")
print()
print(f"  {'Step':>5} {'Multiplier':>12} {'Bar':>20}")
print(f"  {'-'*40}")
for step in [0, 5, 10, 20, 30, 40, 50, 100, 250]:
    t = max(0.0, 1.0 - step / bonus_steps)
    mult = 1.0 + bonus * t
    bar = '#' * int(mult * 2)
    print(f"  {step:5d} {mult:12.1f}×          {bar}")

print(f"\n> Capturing a production-3 planet at step 0 is worth")
print(f"  3 × {cfg.reward.dense_prod_coef} × {1+bonus} = {3 * cfg.reward.dense_prod_coef * (1+bonus):.4f} reward.")
print(f"  At step 100: 3 × {cfg.reward.dense_prod_coef} × 1.0 = {3 * cfg.reward.dense_prod_coef:.4f} reward.")
print(f"  This 10× incentive drives early planet capture.")

In [ ]:
# Compute actual rewards from the live game
env2 = make("orbit_wars", configuration={"seed": 42}, debug=False)
env2.reset(num_agents=2)
prev_states = env2.step([[], []])
prev_obs = prev_states[0].observation
prev_state = parse_observation(prev_obs)

print("=== Actual Dense Relative Rewards (first 10 steps) ===")
print(f"{'Step':>5} {'Reward':>10} {'Ship Gap':>10} {'Prod Gap':>10}")
print("-" * 40)

from v2.reward import _count_all_ships, _count_all_production

for step in range(10):
    moves_p0 = apex_agent(prev_states[0].observation)
    moves_p1 = apex_agent(prev_states[1].observation)
    curr_states = env2.step([moves_p0, moves_p1])

    curr_obs = curr_states[0].observation
    curr_state = parse_observation(curr_obs)
    player = curr_state.player

    r = compute_reward(prev_state, curr_state, player, False, 0.0, cfg.reward)

    all_s = _count_all_ships(curr_state)
    own_s = all_s.get(player, 0)
    best_e_s = max((s for p, s in all_s.items() if p != player), default=0)
    all_p = _count_all_production(curr_state)
    own_p = all_p.get(player, 0)
    best_e_p = max((s for p, s in all_p.items() if p != player), default=0)

    print(f"{curr_state.step:5d} {r:+10.5f} {own_s - best_e_s:+10.0f} {own_p - best_e_p:+10.0f}")

    prev_state = curr_state
    prev_states = curr_states

---

## Part 7: PPO Training — How the Agent Learns

PPO (Proximal Policy Optimization) updates the model using rollout data. Each
transition stores:

| Field | Shape | Description |
|-------|-------|-------------|
| `planet_features` | [N, 40, 22] | Planet matrix at this step |
| `global_features` | [N, 8] | Global context |
| `planet_mask` | [N, 40] | Which planets exist |
| `own_mask` | [N, 40] | Which planets we own |
| `target_indices` | [N, 40] | Sampled action per planet |
| `log_prob` | [N] | Sum of per-planet log probs |
| `returns` | [N] | GAE-computed returns |
| `advantages` | [N] | GAE advantages |
| `values` | [N] | Value predictions |

**Key difference from V1:** Each transition = one game step (not one planet decision).
The log probability is the **sum** across all owned planets.

In [ ]:
print("=== PPO Loss Anatomy ===")
print()
print("1. POLICY LOSS (clipped surrogate):")
print("   ratio = exp(new_log_prob - old_log_prob)")
print("   clipped_ratio = clip(ratio, 1-ε, 1+ε)")
print(f"   ε = {cfg.ppo.clip_coef}")
print("   loss = max(-advantage * ratio, -advantage * clipped_ratio)")
print()
print("2. VALUE LOSS (clipped):")
print(f"   coef = {cfg.ppo.vf_coef}")
print("   loss = 0.5 * max((return - value)², (return - clipped_value)²)")
print()
print("3. ENTROPY BONUS:")
print(f"   coef = {cfg.ppo.ent_coef}")
print("   Encourages exploration by penalizing low-entropy (overconfident) policies")
print()
print("Total loss = policy_loss + vf_coef * value_loss - ent_coef * entropy")

In [ ]:
# Demonstrate GAE (Generalized Advantage Estimation)
gamma = cfg.ppo.gamma  # 0.99
lam = cfg.ppo.gae_lambda  # 0.95

# Synthetic example: 5-step trajectory
rewards =  [0.01, 0.0, -0.02, 0.0, 1.0]  # small dense + terminal win
values =   [0.1,  0.15, 0.12, 0.08, 0.5]  # model's value estimates
dones =    [0,    0,    0,    0,    1]      # terminal at step 5
next_val = 0.0  # bootstrapped value (0 because episode ended)

print("=== GAE Computation (5-step example) ===")
print(f"  gamma={gamma}, lambda={lam}")
print()

gae = 0.0
advantages = [0.0] * 5
returns = [0.0] * 5

for t in reversed(range(5)):
    non_terminal = 1.0 - dones[t]
    if t == 4:
        nv = next_val * non_terminal
    else:
        nv = values[t + 1] * non_terminal

    delta = rewards[t] + gamma * nv - values[t]
    gae = delta + gamma * lam * non_terminal * gae
    advantages[t] = gae
    returns[t] = gae + values[t]

print(f"{'t':>3} {'reward':>8} {'value':>8} {'delta':>8} {'adv':>8} {'return':>8}")
print("-" * 50)
for t in range(5):
    nv = values[t+1] if t < 4 else next_val
    delta = rewards[t] + gamma * nv * (1 - dones[t]) - values[t]
    print(f"{t:3d} {rewards[t]:+8.3f} {values[t]:8.3f} {delta:+8.3f} "
          f"{advantages[t]:+8.3f} {returns[t]:+8.3f}")

print(f"\n> Advantage > 0: action was BETTER than expected")
print(f"  Advantage < 0: action was WORSE than expected")
print(f"  PPO increases probability of positive-advantage actions")

In [ ]:
# Run a mini PPO update to show the training loop
from v2.env import V2OrbitWarsEnv
from v2.train import collect_rollout
from v2.ppo import v2_ppo_update
from src.opponents import ApexOpponent

model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.ppo.lr)

# Short rollout
mini_cfg = load_v2_config('configs/v2_default.yaml')
mini_cfg.ppo.num_envs = 1
mini_cfg.ppo.rollout_steps = 8
mini_cfg.ppo.epochs = 2
mini_cfg.ppo.minibatch_size = 4

opponent = ApexOpponent()
mini_envs = [V2OrbitWarsEnv(mini_cfg, opponent)]
mini_feat = [mini_envs[0].reset(seed=42)]

print("=== Mini Training Step ===")
print(f"  1 env, 8 rollout steps, 2 PPO epochs\n")

# Collect rollout
batch, mini_feat, _, stats = collect_rollout(
    mini_envs, mini_feat, model, mini_cfg, device, 100
)

print(f"Rollout collected:")
print(f"  Samples: {int(stats['samples'])}")
print(f"  Episodes finished: {int(stats['episodes_finished'])}")
print(f"  Episode reward mean: {stats['episode_reward_mean']:+.4f}")
print(f"  Batch shapes: planet_features={batch.planet_features.shape}, "
      f"target_indices={batch.target_indices.shape}")
print()

# PPO update
metrics = v2_ppo_update(
    model, optimizer, batch,
    clip_coef=mini_cfg.ppo.clip_coef,
    ent_coef=mini_cfg.ppo.ent_coef,
    vf_coef=mini_cfg.ppo.vf_coef,
    max_grad_norm=mini_cfg.ppo.max_grad_norm,
    epochs=mini_cfg.ppo.epochs,
    minibatch_size=mini_cfg.ppo.minibatch_size,
    device=device,
)

print(f"PPO update complete:")
print(f"  Total loss:   {metrics['loss']:.4f}")
print(f"  Policy loss:  {metrics['policy_loss']:.4f}")
print(f"  Value loss:   {metrics['value_loss']:.4f}")
print(f"  Entropy:      {metrics['entropy']:.4f}")

model.eval()

---

## Part 8: Mixed Self-Play Curriculum

Training against a fixed opponent (apex) has diminishing returns — the agent
overfits to apex's style. The `V2MixedScheduler` implements a curriculum:

```
Early training:  100% rule-based (apex)     → learn fundamentals
Mid training:     50% apex + 50% self-play  → diversify
Late training:    20% apex + 80% self-play  → compete with self
```

Optionally, 30% of episodes can be 4-player games, adding another dimension
of complexity (diplomacy, kingmaking, three-way fights).

In [ ]:
print(f"=== V2MixedScheduler Configuration ===")
print(f"  four_player_prob:      {cfg.four_player_prob}")
print(f"  rule_based_prob_start: {cfg.rule_based_prob_start} (100% apex at start)")
print(f"  rule_based_prob_end:   {cfg.rule_based_prob_end} (20% apex at end)")
print(f"  rule_based_decay:      {cfg.rule_based_decay_updates} updates")
print()

print(f"Rule-based probability schedule:")
print(f"{'Update':>8} {'Rule-Based%':>12} {'Self-Play%':>12}  {'Bar'}")
print("-" * 50)
for update in [0, 200, 500, 1000, 1500, 2000, 2500, 3000]:
    frac = min(1.0, update / cfg.rule_based_decay_updates)
    rb = cfg.rule_based_prob_start + frac * (cfg.rule_based_prob_end - cfg.rule_based_prob_start)
    sp = 1.0 - rb
    rb_bar = '#' * int(rb * 30)
    sp_bar = '=' * int(sp * 30)
    print(f"{update:8d} {rb:11.0%} {sp:11.0%}   {rb_bar}{sp_bar}")

print(f"\n  # = rule-based (apex)")
print(f"  = = self-play (copy of learning agent)")

### 8.1 Self-Play Weight Syncing

The self-play opponent (`V2SelfPlayOpponent`) maintains a **separate copy** of
OrbitNet. Every `self_play_update_interval` updates (default: 50), the training
model's weights are copied to the self-play opponent.

This creates a **lagged opponent**: the agent always plays against a slightly
older version of itself. This prevents the degenerate equilibrium where the
agent and opponent co-evolve toward trivial strategies.

### 8.2 Why This Curriculum Works

```
Phase 1 (updates 0-500):    "Go to school"
  → 100% apex: Learn basic planet capture, fleet management, sun avoidance
  → Apex is a strong teacher that punishes bad play immediately

Phase 2 (updates 500-2000): "Spar with friends"
  → Gradual shift to self-play: Discover strategies apex doesn't use
  → Still some apex games to prevent forgetting fundamentals

Phase 3 (updates 2000+):    "Compete"
  → 80% self-play: Refine against own-level opponents
  → 20% apex: Maintain baseline competence
```

---

## Part 9: End-to-End Summary

Let's trace the complete data flow one final time.

In [ ]:
print("=" * 70)
print("V2 ORBITNET: COMPLETE DATA FLOW")
print("=" * 70)
print()
print("1. OBSERVATION")
print(f"   Kaggle obs -> parse_observation() -> GameState")
print(f"   {len(state.planets)} planets, {len(state.fleets)} fleets, step={state.step}")
print()
print("2. FLEET PREDICTION")
print(f"   predict_fleet_destination() for each fleet")
print(f"   -> compute_incoming_fleets(): ships & ETA per planet per team")
print()
print("3. FEATURE ENCODING")
print(f"   encode_features() -> V2Features")
print(f"   planet_features: [40, 22]  = {40*22} numbers")
print(f"   global_features: [8]       = 8 numbers")
print(f"   planet_mask:     [40] bool  ({features.planet_mask.sum()} active)")
print(f"   own_mask:        [40] bool  ({features.own_mask.sum()} owned)")
print()
print("4. ORBITNET FORWARD PASS (single pass!)")
print(f"   Planet embed:    [40, 22] -> [40, 128]")
print(f"   + Global embed:  [8] -> [128] (broadcast)")
print(f"   Self-attention:  3 layers × 4 heads")
print(f"   Pairwise head:   [40, 128] -> [40, 40] pair + [40, 1] hold")
print(f"   -> logits:       [40, 41]")
print(f"   -> value:        scalar")
print()
print("5. ACTION")
print(f"   Training:  sample_actions() -> 1 target per owned planet")
print(f"   Eval:      decode_actions() -> multi-target (P > {cfg.env.allocation_threshold})")
print(f"   Ship fraction = softmax probability")
print(f"   Sun avoidance + orbit intercept applied to angles")
print()
print("6. LEARNING")
print(f"   Reward:     dense_relative (ship gap + prod gap)")
print(f"   Algorithm:  PPO (clipped surrogate + value clipping)")
print(f"   Curriculum: apex -> mixed self-play")
print()
print(f"Total model parameters: {count_params(model):,}")
print(f"Forward passes per game step: 1 (vs N in V1)")

---

## Part 10: V1 vs V2 — Architecture Comparison

Both agents solve the same problem with fundamentally different approaches.

In [ ]:
print("+" + "=" * 70 + "+")
print("|                  V1 vs V2 ARCHITECTURE COMPARISON                  |")
print("+" + "=" * 70 + "+")
print()

comparisons = [
    ("Processing",
     "Sequential (N passes per step)",
     "Simultaneous (1 pass per step)"),
    ("Parameters",
     "~493K",
     f"~{count_params(model)//1000}K"),
    ("Action space",
     "Target + 5-way fraction",
     "40x41 allocation matrix"),
    ("Ship fraction",
     "Discrete bins (0.2-1.0)",
     "Continuous (softmax prob)"),
    ("Multi-target",
     "No (one target per planet)",
     "Yes (eval: all P > threshold)"),
    ("Cross-planet coord",
     "Sequential transit updates",
     "Self-attention (parallel)"),
    ("Context per source",
     "Global + KNN + targets",
     "All planets via attention"),
    ("Output head",
     "Per-token linear",
     "Pairwise MLP (src x tgt)"),
]

print(f"{'Aspect':>20} | {'V1 (TransformerPolicy)':>30} | {'V2 (OrbitNet)':>30}")
print("-" * 85)
for aspect, v1, v2 in comparisons:
    print(f"{aspect:>20} | {v1:>30} | {v2:>30}")

print()
print("V2 STRENGTHS:")
print("  + Faster inference (1 pass vs N)")
print("  + Natural multi-target allocation")
print("  + Cross-planet coordination via attention")
print("  + Continuous ship fractions (more expressive)")
print()
print("V2 TRADE-OFFS:")
print("  - Larger output space (40x41 vs ~31 per planet)")
print("  - No sequential transit updates (relies on attention)")
print("  - Pairwise head is O(P²) in planet count")

---

## Key Takeaways

1. **One pass, all planets.** OrbitNet processes every planet simultaneously through
   self-attention, outputting a 40×41 allocation matrix in a single forward pass.

2. **Softmax = fraction.** The probability of sending to a target IS the fraction
   of ships to send. This elegant unification simplifies the action space.

3. **Pairwise > independent.** The pairwise output head captures source-target
   relationships (distance, relative strength, threat level) that a per-planet
   head cannot.

4. **Attention = coordination.** Self-attention lets owned planets coordinate
   without sequential processing. Planet A knows Planet B's situation before
   deciding where to send ships.

5. **Dense relative reward.** Rewarding advantage over the best enemy (not absolute
   accumulation) teaches attacking when profitable. Early production bonus drives
   land-grab strategy.

6. **Train:** `uv run python -m v2.train --config configs/v2_default.yaml`